# Gerador de Dados - T1 IA
Este notebook prepara o dataset conforme as exigências do enunciado T1:
1. Carrega e limpa o dataset original UCI.
2. Transforma o problema binário original em um problema de 5 classes.
3. Gera estados intermediários ("Tem jogo" e "Possibilidade de Fim de Jogo") que não existem no original.
4. Garante um dataset balanceado (sugestão de 200+ amostras por classe).

In [2]:
import os
import random
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### 1. Definição da Lógica de Classificação
Como o dataset UCI só possui rótulos de vitória, precisamos de uma função que identifique os 5 estados exigidos pelo trabalho.

In [3]:
def verificar_estado(tab):
    vitorias = [[0,1,2], [3,4,5], [6,7,8], [0,3,6], [1,4,7], [2,5,8], [0,4,8], [2,4,6]]
    # Verificação de vitória
    for v in vitorias:
        soma = tab[v[0]] + tab[v[1]] + tab[v[2]]
        if soma == 3: return 'X vence'
        if soma == -3: return 'O vence'
    
    # Verificação de Possibilidade de Fim de Jogo (Ameaça/Oportunidade)
    for v in vitorias:
        soma = tab[v[0]] + tab[v[1]] + tab[v[2]]
        if abs(soma) == 2 and 0 in [tab[v[0]], tab[v[1]], tab[v[2]]]:
            return 'Possibilidade de Fim de Jogo'
    
    # Empate ou Em andamento
    if 0 not in tab: return 'Empate'
    return 'Tem jogo'

### 2. Carregamento e Limpeza (Dados Originais)
Aqui removemos a codificação original da UCI ('positive'/'negative') e aplicamos a nossa escala numérica e novas classes.

In [4]:
def processar_dados_originais(caminho):
    if not os.path.exists(caminho):
        print(f"Aviso: Arquivo {caminho} não encontrado.")
        return pd.DataFrame()
    
    # O CSV original não tem cabeçalho e os dados estão em x, o, b
    df = pd.read_csv(caminho, header=None)
    
    # Mapeamento para valores numéricos
    mapping = {'x': 1, 'o': -1, 'b': 0}
    for i in range(9):
        df[i] = df[i].map(mapping)
    
    # Removemos a 10ª coluna (rótulo original da UCI) pois ela é insuficiente (binária)
    # Recalculamos o target com base nas 5 classes exigidas
    tabuleiros = df.iloc[:, :9].values.tolist()
    novos_targets = [verificar_estado(t) for t in tabuleiros]
    
    df_limpo = pd.DataFrame(tabuleiros, columns=[f'pos{i}' for i in range(9)])
    df_limpo['target'] = novos_targets
    
    return df_limpo

df_original = processar_dados_originais('../data/raw/tic-tac-toe.data')
print("Distribuição após limpar dados originais:")
print(df_original['target'].value_counts())

Distribuição após limpar dados originais:
target
X vence    626
O vence    316
Empate      16
Name: count, dtype: int64


### 3. Complementação Sintética e Balanceamento
Como os dados originais focam apenas em finais de jogo, simulamos partidas para obter exemplos de 'Tem jogo' e 'Possibilidade de Fim de Jogo', garantindo o balanceamento.

In [5]:
def simular_estados(n_por_classe, dados_existentes):
    classes = ['X vence', 'O vence', 'Empate', 'Possibilidade de Fim de Jogo', 'Tem jogo']
    # Inicializa com o que já temos dos dados originais
    repositorio = {c: [] for c in classes}
    for _, row in dados_existentes.iterrows():
        if len(repositorio[row['target']]) < n_por_classe:
            repositorio[row['target']].append(row.tolist())
    
    # Simula partidas para completar o que falta
    while any(len(v) < n_por_classe for v in repositorio.values()):
        tab = [0] * 9
        jogadores = [1, -1]
        random.shuffle(jogadores)
        jog = jogadores[0]
        pos = list(range(9))
        random.shuffle(pos)
        
        for p in pos:
            tab[p] = jog
            est = verificar_estado(tab)
            if len(repositorio[est]) < n_por_classe:
                repositorio[est].append(tab.copy() + [est])
            if est in ['X vence', 'O vence', 'Empate']: break
            jog *= -1
            
    lista_final = []
    for c in repositorio: lista_final.extend(repositorio[c])
    return pd.DataFrame(lista_final, columns=[f'pos{i}' for i in range(9)] + ['target'])

# O PDF sugere 200 amostras, vamos usar 1000 para maior robustez, mantendo o equilíbrio
df_final = simular_estados(n_por_classe=1000, dados_existentes=df_original)
print("\nDistribuição final balanceada:")
print(df_final['target'].value_counts())


Distribuição final balanceada:
target
X vence                         1000
O vence                         1000
Empate                          1000
Possibilidade de Fim de Jogo    1000
Tem jogo                        1000
Name: count, dtype: int64


In [6]:
# Divisão Estratificada (Treino, Validação, Teste)
X = df_final.drop(columns=['target'])
y = df_final['target']

x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
x_test, x_val, y_test, y_val = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

train_df = pd.concat([x_train, y_train], axis=1)
val_df = pd.concat([x_val, y_val], axis=1)
test_df = pd.concat([x_test, y_test], axis=1)

path_processed = '../data/processed'
train_df.to_csv(f'{path_processed}/train.csv', index=False)
val_df.to_csv(f'{path_processed}/val.csv', index=False)
test_df.to_csv(f'{path_processed}/test.csv', index=False)
df_final.to_csv(f'{path_processed}/tic_tac_toe_processed.csv', index=False)
df_original.to_csv(f'{path_processed}/tic_tac_toe_original_limpo.csv', index=False)

# Scaler
scaler = StandardScaler()
scaler.fit(x_train)
joblib.dump(scaler, '../models/scaler.pkl')

print(f"Sucesso! Dataset de treino com {train_df.shape[0]} amostras.")

Sucesso! Dataset de treino com 3500 amostras.
